# Notebook 3: Performance Tuning, Shuffle Mechanics & Cloud Storage (S3) (Student Lab)
### Hands-on Workshop: Apache Spark Foundation & Ingestion Framework (Day 2 Morning)
### Related Presentation Slides: Slides 17 - 19 (Module 5) and Slides 21 - 22 (Module 6)

---

## Learning Objectives:
1. Demystify Shuffle: network I/O cost during `join` and `groupBy` (Slide 17).
2. Master Broadcast Hash Join (`F.broadcast`) for dimension/lookup tables (Slide 16).
3. Solve The Small File Problem on AWS S3 / Data Lake storage (Slide 18).
4. Compare `coalesce(N)` (no shuffle) vs `repartition(N)` (full shuffle) (Slide 19).
5. Implement partitioned Parquet writes with partition pruning (Slide 21).


In [ ]:
# Setup environment & SparkSession
!pip install -q pyspark

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
import time, os

spark = SparkSession.builder \
    .appName("03_Performance_and_Storage") \
    .master("local[*]") \
    .config("spark.sql.adaptive.enabled", "true") \
    .getOrCreate()

print(f"Spark initialized with Adaptive Query Execution (AQE): {spark.conf.get('spark.sql.adaptive.enabled')}")


---
## Step 1: Distributed Joins & The Cost of Shuffle (Related: Slide 16 & Slide 17)

When joining two tables:
* If both are large, Spark must hash and send rows over the physical network so matching keys land on the same worker node (Shuffle Sort-Merge Join). Network I/O is the slowest operation in a cluster (Slide 17).
* Broadcast Join Optimization: If one table is small (e.g., team financials lookup < 100 MB), the Driver broadcasts a complete copy to every executor.
  * Result: Zero network shuffle of the large events table!


In [ ]:
df_events = spark.read.json("data/raw/bundesliga_events.json") \
    .select(F.col("event_id"), F.col("team.name").alias("team_name"), F.col("minute"))

df_financials = spark.read.option("header", "true").option("inferSchema", "true").csv("data/raw/team_financials.csv")

print("Financials Lookup Table:")
df_financials.show()


In [ ]:
# Scenario A: Standard Join (Spark Catalyst decides, may involve shuffle exchange)
df_joined_standard = df_events.join(df_financials, on="team_name", how="inner")
print("--- Standard Join Plan ---")
df_joined_standard.explain()


In [ ]:
# Scenario B: Explicit Broadcast Join (Slide 16)
df_joined_broadcast = df_events.join(F.broadcast(df_financials), on="team_name", how="inner")
print("--- Broadcast Join Plan (Notice BroadcastHashJoin & no ShuffleExchange) ---")
df_joined_broadcast.explain()


---
## Step 2: The Small File Problem on AWS S3 / Lakehouse (Related: Slide 18)

> Important Note (Slide 18):
> In Spark, 1 output file is created per partition.
> If your DataFrame has 200 partitions, Spark writes 200 files.
> If each file is only 5 KB, you create The Small File Problem:
> * Enormous latency on downstream reads (opening 10,000 files takes seconds).
> * High cloud storage API charges (AWS S3 charges per 1,000 GET/LIST requests).
> 
> Target File Size: Aim for 128 MB – 256 MB per Parquet file.


---
## Step 3: coalesce() vs repartition() (Related: Slide 19)

| Operation | Shuffle? | When to Use |
| :--- | :--- | :--- |
| `coalesce(N)` | No Shuffle | Downsizing partition count before writing to disk (e.g. from 20 to 2). Fast & zero network cost. |
| `repartition(N)` | Full Shuffle | Increasing partition count, or redistributing skewed keys evenly across workers. |


In [ ]:
df_large = spark.range(1, 100_000, numPartitions=16)
print(f"Initial partition count: {df_large.rdd.getNumPartitions()}")

# Test coalesce (reduces to 2 partitions WITHOUT shuffle)
df_coalesced = df_large.coalesce(2)
print(f"Coalesced partition count: {df_coalesced.rdd.getNumPartitions()}")

# Test repartition (forces full shuffle)
df_repartitioned = df_large.repartition(4)
print(f"Repartitioned partition count: {df_repartitioned.rdd.getNumPartitions()}")


---
## Step 4: Writing to Partitioned Parquet (Related: Slide 21)

In production Data Lakehouses:
1. Always control the partition count before saving (`.coalesce(2)`).
2. Partition the disk directory layout using high-cardinality filters (e.g., `partitionBy("team_name")`).


In [ ]:
output_path = "data/processed/bundesliga_parquet"

# Write optimized Parquet
df_joined_broadcast.coalesce(2) \
    .write \
    .mode("overwrite") \
    .partitionBy("team_name") \
    .parquet(output_path)

print(f"Data written to {output_path}")

# Inspect physical disk directory structure
print("\nGenerated disk folders:")
for root, dirs, files in os.walk(output_path):
    level = root.replace(output_path, '').count(os.sep)
    indent = ' ' * 4 * level
    print(f"{indent}{os.path.basename(root)}/")
    subindent = ' ' * 4 * (level + 1)
    for f in files:
        if f.endswith(".parquet"):
            size_kb = round(os.path.getsize(os.path.join(root, f)) / 1024, 1)
            print(f"{subindent}{f} ({size_kb} KB)")


---
## Step 5: Partition Pruning in Action (Related: Slide 21)

When querying partitioned Parquet, Spark skips reading non-matching folders entirely (Slide 21):


In [ ]:
# Reading back with Partition Pruning:
df_pruned = spark.read.parquet(output_path).filter(F.col("team_name") == "Bayern Munich")
print("--- Execution plan with PartitionFilters (Pruning) ---")
df_pruned.explain()


---
## Hands-on Lab Exercise 3 (Corresponding to Module 5 - 6 Hands-on)

### Task:
1. Read the Parquet data from `data/processed/bundesliga_parquet`.
2. Group by `team_name` and compute total event count and maximum `minute`.
3. Coalesce the result to exactly 1 partition and save it as Parquet in `data/processed/team_summary_single_file`.


In [ ]:
# TODO: Write your solution below:

# df_summary = ...

# Verification:
# print(f"Summary rows: {df_summary.count()}")
# df_summary.show()
